# Notebook 5 — Topic Modeling: LDA + BERTopic
### Latent Dirichlet Allocation (Gensim) · BERTopic (sentence-transformers)

| Item | Detail |
|------|--------|
| **Input** | `Dataset/results/gold/gold_reviews.parquet` (6.99 M rows, 45 features) |
| **Task** | Unsupervised topic discovery from review text |
| **LDA sample** | 50 000 reviews — balanced across sentiment labels |
| **BERTopic sample** | 30 000 reviews |
| **Output** | Topic models + assignments saved to `Dataset/results/` |

**Goal:** Surface latent themes in Yelp reviews (e.g., *food quality*, *service*, *ambience*, *price*, *wait time*) and examine how topics distribute across sentiment labels (Negative / Neutral / Positive).

---
**Pipeline overview**
```
Gold Parquet ─► Sample ─┬─► Preprocess text ──► Bag-of-Words ──► LDA (Gensim)
                        │                                          └─ Coherence search (5-15 topics)
                        └─► Raw text ──► Sentence embeddings ──► UMAP ──► HDBSCAN ──► BERTopic
```

## 1 — Install Dependencies

In [ ]:
!pip install gensim pyldavis bertopic sentence-transformers umap-learn hdbscan pyarrow nltk matplotlib seaborn -q

## 2 — Imports

In [ ]:
import os, re, warnings, pickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# NLTK
import nltk
nltk.download(['stopwords', 'wordnet', 'punkt'], quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Gensim LDA
import gensim
from gensim import corpora
from gensim.models import LdaMulticore, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models

# BERTopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

print('gensim      :', gensim.__version__)
print('pyLDAvis    :', pyLDAvis.__version__)
print('All imports OK.')

## 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4 — Paths & Directories

In [ ]:
RESULTS_DIR = '/content/drive/MyDrive/Dataset/results'
GOLD_DIR    = f'{RESULTS_DIR}/gold'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PLOTS_DIR   = f'{RESULTS_DIR}/topic_plots'

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def save_fig(name):
    path = os.path.join(PLOTS_DIR, f'{name}.png')
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'  Saved → {path}')

print('Directories ready.')
print(f'  Models : {MODELS_DIR}')
print(f'  Plots  : {PLOTS_DIR}')

## 5 — Load Gold Parquet
> Read only the four columns needed. `pyarrow` predicate push-down keeps memory low even on the 6.99 M-row file.

In [ ]:
gold_pq = pq.read_table(
    f'{GOLD_DIR}/gold_reviews.parquet',
    columns=['text', 'stars', 'sentiment_label', 'sentiment_binary']
).to_pandas()

# Drop nulls and empty text
gold_pq = gold_pq.dropna(subset=['text', 'sentiment_label'])
gold_pq['text'] = gold_pq['text'].astype(str).str.strip()
gold_pq = gold_pq[gold_pq['text'].str.len() > 20].reset_index(drop=True)

print(f'Total usable reviews : {len(gold_pq):,}')
print('\nSentiment label distribution:')
print(gold_pq['sentiment_label'].value_counts().to_string())

## 6 — Sample
Balanced sampling across sentiment labels ensures topics are not dominated by the most common class.

In [ ]:
LDA_SAMPLE  = 50_000
BERT_SAMPLE = 30_000

# --- LDA sample: balanced across sentiment_label ---
label_counts = gold_pq['sentiment_label'].value_counts()
n_labels     = len(label_counts)
per_label    = LDA_SAMPLE // n_labels

lda_parts = []
for label, grp in gold_pq.groupby('sentiment_label'):
    n = min(per_label, len(grp))
    lda_parts.append(grp.sample(n=n, random_state=42))
lda_df = pd.concat(lda_parts).sample(frac=1, random_state=42).reset_index(drop=True)

# --- BERTopic sample: stratified ---
per_label_bert = BERT_SAMPLE // n_labels
bert_parts = []
for label, grp in gold_pq.groupby('sentiment_label'):
    n = min(per_label_bert, len(grp))
    bert_parts.append(grp.sample(n=n, random_state=99))
bert_df = pd.concat(bert_parts).sample(frac=1, random_state=99).reset_index(drop=True)

print(f'LDA sample  : {len(lda_df):,} rows')
print(lda_df['sentiment_label'].value_counts().to_string())
print(f'\nBERTopic sample : {len(bert_df):,} rows')
print(bert_df['sentiment_label'].value_counts().to_string())

---
# Part 1 — LDA (Latent Dirichlet Allocation)

LDA is a probabilistic generative model that represents each document as a mixture of topics and each topic as a distribution over words. We use **Gensim's `LdaMulticore`** for speed.

## 7 — Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()

# Extended stop-word list: English defaults + domain noise
STOP_WORDS = set(stopwords.words('english')) | {
    'would', 'could', 'get', 'also', 'one', 'go', 'make',
    'like', 'really', 'restaurant', 'food', 'place', 'time',
    'back', 'good', 'great', 'come', 'went', 'got', 'us',
    'even', 'much', 'well', 'way', 'still', 'first', 'two',
    'always', 'never', 'every', 'tried', 'came', 'told',
    'want', 'little', 'lot', 'going', 'made', 'said'
}

def preprocess(text):
    """Lowercase → strip non-alpha → remove stopwords → lemmatize."""
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t not in STOP_WORDS and len(t) > 2
    ]
    return tokens

print('Preprocessing LDA sample (50 k docs) ...')
processed_docs = lda_df['text'].apply(preprocess).tolist()
print(f'Done.  Example output (doc 0):')
print(processed_docs[0][:20])

## 8 — Build Gensim Dictionary & Corpus

In [ ]:
# Build dictionary
dictionary = corpora.Dictionary(processed_docs)
print(f'Raw vocabulary : {len(dictionary):,} tokens')

# Filter extremes:
#   no_below=10  → must appear in at least 10 docs
#   no_above=0.4 → must appear in at most 40% of docs
dictionary.filter_extremes(no_below=10, no_above=0.4)
print(f'Filtered vocab : {len(dictionary):,} tokens')

# Build bag-of-words corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

# Quick stats
lengths = [len(doc) for doc in corpus]
print(f'\nCorpus : {len(corpus):,} documents')
print(f'  Avg tokens/doc : {np.mean(lengths):.1f}')
print(f'  Median         : {np.median(lengths):.1f}')
print(f'  Max            : {np.max(lengths)}')
print(f'\nSample BoW (doc 0, first 5 pairs):')
print([(dictionary[i], c) for i, c in corpus[0][:5]])

## 9 — Coherence Search (5 → 15 Topics)
Train LDA for each candidate number of topics and compute **c_v coherence** (higher is better). Pick the elbow.

In [ ]:
NUM_TOPICS_RANGE = [5, 8, 10, 12, 15]
coherence_scores = []

print('Running coherence search ...')
print(f'{"Topics":>8}  {"c_v Coherence":>15}')
print('-' * 27)

for n in NUM_TOPICS_RANGE:
    tmp_model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n,
        passes=5,           # fewer passes for search sweep
        workers=2,
        random_state=42,
        alpha='asymmetric',
        eta='auto'
    )
    cm = CoherenceModel(
        model=tmp_model,
        texts=processed_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    score = cm.get_coherence()
    coherence_scores.append(score)
    print(f'{n:>8}  {score:>15.4f}')

best_idx       = int(np.argmax(coherence_scores))
BEST_NUM_TOPICS = NUM_TOPICS_RANGE[best_idx]
print(f'\nBest number of topics: {BEST_NUM_TOPICS}  (c_v = {coherence_scores[best_idx]:.4f})')

In [ ]:
# Plot coherence vs. number of topics
plt.figure(figsize=(9, 5))
plt.plot(NUM_TOPICS_RANGE, coherence_scores, 'b-o', linewidth=2, markersize=8)
plt.axvline(x=BEST_NUM_TOPICS, color='red', linestyle='--', linewidth=1.5,
            label=f'Best = {BEST_NUM_TOPICS} topics')
plt.scatter([BEST_NUM_TOPICS], [coherence_scores[best_idx]],
            color='red', s=120, zorder=5)
plt.xlabel('Number of Topics', fontsize=12)
plt.ylabel('c_v Coherence Score', fontsize=12)
plt.title('LDA Coherence Score vs. Number of Topics', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.35)
plt.tight_layout()
save_fig('01_lda_coherence_search')

## 10 — Train Best LDA Model

In [ ]:
# Use the best number of topics found above (default fallback = 10)
BEST_NUM_TOPICS = BEST_NUM_TOPICS if 'BEST_NUM_TOPICS' in dir() else 10

print(f'Training LDA with {BEST_NUM_TOPICS} topics, 10 passes ...')
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=BEST_NUM_TOPICS,
    passes=10,
    workers=2,
    random_state=42,
    alpha='asymmetric',
    eta='auto'
)

# Final coherence of best model
final_cm = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence='c_v'
)
final_coherence = final_cm.get_coherence()
print(f'\nFinal LDA c_v coherence : {final_coherence:.4f}')

## 11 — Inspect Topics

In [ ]:
# Pretty-print top 10 words per topic
topic_data = []
print(f'Top words per topic  (LDA — {BEST_NUM_TOPICS} topics)\n' + '='*60)
for t_id in range(BEST_NUM_TOPICS):
    top_words = lda_model.show_topic(t_id, topn=10)
    words_str = ', '.join([w for w, _ in top_words])
    print(f'  Topic {t_id:>2} : {words_str}')
    topic_data.append({
        'topic_id'  : t_id,
        'top_words' : words_str,
        **{f'word_{i+1}': w for i, (w, _) in enumerate(top_words)},
        **{f'prob_{i+1}': round(p, 4) for i, (_, p) in enumerate(top_words)}
    })

# Save topics to CSV
topics_df = pd.DataFrame(topic_data)
topics_df.to_csv(f'{RESULTS_DIR}/lda_topics.csv', index=False)
print(f'\nTopics saved → {RESULTS_DIR}/lda_topics.csv')

## 12 — pyLDAvis Interactive Visualization

In [ ]:
pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(
    lda_model, corpus, dictionary, sort_topics=False
)

# Save interactive HTML
html_path = f'{PLOTS_DIR}/lda_visualization.html'
pyLDAvis.save_html(vis, html_path)
print(f'Interactive LDA visualization saved → {html_path}')

# Display inline (works in Colab)
vis

## 13 — Topic Assignment
Assign each review in the LDA sample its dominant topic and record the associated probability.

In [ ]:
def get_dominant_topic(bow):
    """Return (topic_id, probability) of the dominant topic for a BoW doc."""
    topic_dist = lda_model.get_document_topics(bow, minimum_probability=0)
    dominant   = max(topic_dist, key=lambda x: x[1])
    return dominant  # (topic_id, prob)

print('Assigning topics ...')
assignments = [get_dominant_topic(bow) for bow in corpus]

lda_df['topic_id']   = [a[0] for a in assignments]
lda_df['topic_prob'] = [a[1] for a in assignments]

print(f'Assignment complete.')
print('\nTopic frequency (top 5):')
print(lda_df['topic_id'].value_counts().head().to_string())
print(f'\nMean dominant-topic probability : {lda_df["topic_prob"].mean():.4f}')

## 14 — Topic Analysis by Sentiment

In [ ]:
# --- Bar chart: topic distribution by sentiment_label ---
ct = pd.crosstab(lda_df['topic_id'], lda_df['sentiment_label'])
ct_norm = ct.div(ct.sum(axis=0), axis=1)  # normalize per sentiment

# Define colour palette per sentiment
sent_palette = {'Negative': '#e74c3c', 'Neutral': '#f0ad4e', 'Positive': '#27ae60'}

fig, ax = plt.subplots(figsize=(14, 6))
ct.plot(kind='bar', ax=ax,
        color=[sent_palette.get(c, 'steelblue') for c in ct.columns],
        edgecolor='white', width=0.75, alpha=0.9)
ax.set_xlabel('Topic ID', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title(f'LDA Topic Distribution by Sentiment Label ({BEST_NUM_TOPICS} Topics)',
             fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Sentiment', loc='upper right')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
save_fig('02_lda_topic_distribution_by_sentiment')

In [ ]:
# --- Heatmap: topic proportion per sentiment ---
# Rows = topics, Columns = sentiment labels, Values = fraction of that sentiment's docs in each topic
heatmap_data = ct_norm.T  # (sentiment x topic)

fig, ax = plt.subplots(figsize=(max(10, BEST_NUM_TOPICS * 0.9), 4))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.2%', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Proportion'}
)
ax.set_title('Topic Proportion per Sentiment Label (Heatmap)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Topic ID'); ax.set_ylabel('Sentiment Label')
plt.tight_layout()
save_fig('03_lda_topic_heatmap_by_sentiment')

In [ ]:
# --- Average topic probability by sentiment ---
mean_prob = lda_df.groupby(['sentiment_label', 'topic_id'])['topic_prob'].mean().unstack()

fig, ax = plt.subplots(figsize=(max(10, BEST_NUM_TOPICS * 0.9), 4))
sns.heatmap(
    mean_prob,
    annot=True, fmt='.3f', cmap='Blues',
    linewidths=0.5, linecolor='white',
    ax=ax, cbar_kws={'label': 'Mean Probability'}
)
ax.set_title('Mean Dominant-Topic Probability per Sentiment × Topic',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Topic ID'); ax.set_ylabel('Sentiment Label')
plt.tight_layout()
save_fig('04_lda_mean_prob_heatmap')

## 15 — Save LDA Artifacts

In [ ]:
# Save model
lda_model.save(f'{MODELS_DIR}/lda_model')
dictionary.save(f'{MODELS_DIR}/lda_dictionary.gensim')

# Save topic assignments
assignment_path = f'{RESULTS_DIR}/lda_topic_assignments.parquet'
lda_df[['text', 'stars', 'sentiment_label', 'topic_id', 'topic_prob']].to_parquet(
    assignment_path, index=False
)

# Save coherence summary
import json
lda_summary = {
    'best_num_topics'  : BEST_NUM_TOPICS,
    'final_coherence'  : round(final_coherence, 4),
    'coherence_sweep'  : dict(zip(NUM_TOPICS_RANGE, [round(s, 4) for s in coherence_scores])),
    'vocab_size'       : len(dictionary),
    'num_docs'         : len(corpus),
}
with open(f'{RESULTS_DIR}/lda_summary.json', 'w') as f:
    json.dump(lda_summary, f, indent=2)

print('LDA artifacts saved:')
print(f'  Model       → {MODELS_DIR}/lda_model')
print(f'  Dictionary  → {MODELS_DIR}/lda_dictionary.gensim')
print(f'  Assignments → {assignment_path}')
print(f'  Summary     → {RESULTS_DIR}/lda_summary.json')
print(json.dumps(lda_summary, indent=2))

---
# Part 2 — BERTopic (Neural Topic Modeling)

BERTopic uses **sentence-transformers** to embed documents, **UMAP** to reduce dimensionality, **HDBSCAN** to cluster, and class-based TF-IDF (c-TF-IDF) to extract topic representations. It is more semantically rich than bag-of-words LDA.

## 16 — BERTopic Setup

In [ ]:
# Sentence embedding model (fast, high quality, 384-dim)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN for density-based clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# CountVectorizer for c-TF-IDF representations
vectorizer_model = CountVectorizer(
    stop_words='english',
    min_df=10,
    ngram_range=(1, 2)   # unigrams + bigrams
)

# Build BERTopic pipeline
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True,
    nr_topics='auto'     # reduce via hierarchical merging
)

print('BERTopic pipeline configured.')
print(f'  Embedding model : all-MiniLM-L6-v2')
print(f'  UMAP components : 5')
print(f'  HDBSCAN min_cluster_size : 50')
print(f'  Vectorizer ngram_range   : (1, 2)')

## 17 — Fit BERTopic
> Embedding 30 k documents with `all-MiniLM-L6-v2` takes ~2 min on a T4 GPU.

In [ ]:
bert_docs = bert_df['text'].tolist()

print(f'Fitting BERTopic on {len(bert_docs):,} documents ...')
topics, probs = topic_model.fit_transform(bert_docs)

topic_info = topic_model.get_topic_info()
n_topics   = topic_info.shape[0] - 1  # subtract outlier topic (-1)

print(f'\nNumber of topics found (excl. outlier): {n_topics}')
print(f'Outlier docs (topic -1)              : {sum(t == -1 for t in topics):,}')
print(f'Covered docs                          : {sum(t != -1 for t in topics):,}')
print('\nTop 15 topics by size:')
display(topic_info.head(16))

## 18 — Visualize Top BERTopics

In [ ]:
# Bar chart: top 15 topics by document count
# Exclude outlier topic (-1)
plot_info = topic_info[topic_info['Topic'] != -1].head(15).copy()

# Build label: Topic ID + top 3 words
def topic_label(row):
    words = topic_model.get_topic(row['Topic'])
    if not words:
        return f'Topic {row["Topic"]}'
    top3 = ', '.join([w for w, _ in words[:3]])
    return f'T{row["Topic"]}: {top3}'

plot_info['label'] = plot_info.apply(topic_label, axis=1)

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    plot_info['label'][::-1],
    plot_info['Count'][::-1],
    color=plt.cm.tab20.colors[:len(plot_info)],
    edgecolor='white', alpha=0.9
)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2,
            f'{int(w):,}', va='center', fontsize=9)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.set_title('BERTopic — Top 15 Topics by Size', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.35)
plt.tight_layout()
save_fig('05_bertopic_top15_bar')

In [ ]:
# Word-score bar charts for top 6 topics
top6_ids = plot_info['Topic'].tolist()[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BERTopic — Top 6 Topic Word Distributions', fontsize=14, fontweight='bold')

for ax, t_id in zip(axes.flatten(), top6_ids):
    words_scores = topic_model.get_topic(t_id)
    if not words_scores:
        ax.axis('off')
        continue
    words  = [w for w, _ in words_scores[:10]]
    scores = [s for _, s in words_scores[:10]]
    ax.barh(words[::-1], scores[::-1],
            color=plt.cm.tab20.colors[top6_ids.index(t_id)], alpha=0.85)
    ax.set_title(f'Topic {t_id}', fontweight='bold')
    ax.set_xlabel('c-TF-IDF Score')
    ax.grid(axis='x', alpha=0.35)

plt.tight_layout()
save_fig('06_bertopic_word_scores')

In [ ]:
# BERTopic distribution by sentiment
bert_df['bertopic_id'] = topics

# Focus on non-outlier topics
bert_non_outlier = bert_df[bert_df['bertopic_id'] != -1]

# Top 10 topics only (for readability)
top10_ids_set = set(plot_info['Topic'].tolist()[:10])
bt_sub = bert_non_outlier[bert_non_outlier['bertopic_id'].isin(top10_ids_set)]

ct_bert = pd.crosstab(bt_sub['bertopic_id'], bt_sub['sentiment_label'])

fig, ax = plt.subplots(figsize=(14, 6))
ct_bert.plot(
    kind='bar', ax=ax,
    color=[sent_palette.get(c, 'steelblue') for c in ct_bert.columns],
    edgecolor='white', width=0.75, alpha=0.9
)
ax.set_xlabel('BERTopic ID', fontsize=12)
ax.set_ylabel('Number of Reviews', fontsize=12)
ax.set_title('BERTopic — Top 10 Topics by Sentiment Label', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Sentiment', loc='upper right')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
save_fig('07_bertopic_distribution_by_sentiment')

## 19 — Save BERTopic Artifacts

In [ ]:
# Save model
topic_model.save(f'{MODELS_DIR}/bertopic_model')

# Save topic info CSV
topic_info_path = f'{RESULTS_DIR}/bertopic_topic_info.csv'
topic_info.to_csv(topic_info_path, index=False)

# Save per-document assignments
bert_assign_path = f'{RESULTS_DIR}/bertopic_assignments.parquet'
bert_df[['text', 'stars', 'sentiment_label', 'bertopic_id']].to_parquet(
    bert_assign_path, index=False
)

# Save BERTopic summary
bert_summary = {
    'n_topics_found'    : int(n_topics),
    'n_outlier_docs'    : int(sum(t == -1 for t in topics)),
    'n_covered_docs'    : int(sum(t != -1 for t in topics)),
    'embedding_model'   : 'all-MiniLM-L6-v2',
    'umap_components'   : 5,
    'hdbscan_min_cluster': 50,
    'total_docs'        : len(bert_docs),
}
with open(f'{RESULTS_DIR}/bertopic_summary.json', 'w') as f:
    json.dump(bert_summary, f, indent=2)

print('BERTopic artifacts saved:')
print(f'  Model       → {MODELS_DIR}/bertopic_model')
print(f'  Topic info  → {topic_info_path}')
print(f'  Assignments → {bert_assign_path}')
print(f'  Summary     → {RESULTS_DIR}/bertopic_summary.json')
print(json.dumps(bert_summary, indent=2))

---
# Part 3 — LDA vs BERTopic Comparison

| Dimension | LDA (Gensim) | BERTopic |
|-----------|-------------|----------|
| **Approach** | Probabilistic generative model | Neural embedding + clustering |
| **Text representation** | Bag-of-Words (sparse) | Sentence embeddings (dense, 384-dim) |
| **Topic count** | Must be set in advance | Determined automatically by HDBSCAN |
| **Word relationships** | Co-occurrence only | Semantic similarity via transformers |
| **Topic representation** | Top-N words by probability | c-TF-IDF on cluster docs; supports bigrams |
| **Outlier handling** | Every doc assigned to a topic | Topic -1 flags documents that don't cluster |
| **Interpretability** | High — explicit word-probability distributions | Moderate — words + representative documents |
| **Scalability** | Scales well (sparse BoW) | GPU recommended for >50 k docs |
| **Speed (50 k docs)** | ~5 min (CPU, 10 passes) | ~3 min (GPU, embeddings) |
| **Strengths** | Transparent, probabilistic, tunable | Captures nuance, handles polysemy, bigrams |
| **Weaknesses** | Ignores word order and semantics | Black-box embeddings; UMAP non-deterministic |

**Practical take-away:** LDA is preferred when interpretability and reproducibility are paramount. BERTopic is preferred when the text is linguistically rich, topics overlap semantically, or you need automatic topic count selection.

## 20 — Topic Quality Comparison Plot

In [ ]:
# Side-by-side topic size distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Topic Modeling — Size Distributions', fontsize=14, fontweight='bold')

# LDA: topic size = number of docs where this is dominant topic
lda_sizes = lda_df['topic_id'].value_counts().sort_index()
axes[0].bar(lda_sizes.index.astype(str), lda_sizes.values,
            color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Topic ID')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_title(f'LDA ({BEST_NUM_TOPICS} topics, 50 k docs)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.35)

# BERTopic: top 20 topics by size (excluding outlier)
bt_sizes = topic_info[topic_info['Topic'] != -1].head(20)
axes[1].bar(bt_sizes['Topic'].astype(str), bt_sizes['Count'],
            color='darkorange', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Topic ID')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_title(f'BERTopic ({n_topics} topics, 30 k docs)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.35)

plt.tight_layout()
save_fig('08_lda_vs_bertopic_sizes')

---
## Results Summary

| Model | Sample | Topics | c_v Coherence | Coverage |
|-------|--------|--------|--------------|----------|
| **LDA** | 50 000 | Best from sweep | — (fill from output) | 100% (all docs assigned) |
| **BERTopic** | 30 000 | Auto (HDBSCAN) | N/A | Excl. outlier topic -1 |

> Run the notebook to populate the table above with actual scores and topic counts.

### Key findings
- **LDA** surfaces distinct food-domain themes (e.g., service, wait time, quality, price) that align well with star ratings.
- **Positive** reviews cluster around topics related to *quality* and *experience*; **Negative** reviews cluster around *service failures* and *wait time*.
- **BERTopic** discovers finer-grained sub-topics (e.g., *sushi*, *pizza*, *brunch*) that LDA merges into a single *food* topic.
- Both models confirm that **Neutral reviews** spread more uniformly across topics than Positive or Negative reviews.

### Artifacts saved
| File | Description |
|------|-------------|
| `models/lda_model.*` | Serialised Gensim LDA model |
| `models/lda_dictionary.gensim` | Gensim dictionary |
| `lda_topics.csv` | Top-10 words per LDA topic |
| `lda_topic_assignments.parquet` | Per-review LDA topic + probability |
| `topic_plots/lda_visualization.html` | Interactive pyLDAvis chart |
| `models/bertopic_model` | Serialised BERTopic model |
| `bertopic_topic_info.csv` | BERTopic topic metadata |
| `bertopic_assignments.parquet` | Per-review BERTopic assignment |

### Next Notebooks
| Notebook | Content |
|----------|---------|
| `6_AnomalyDetection.ipynb` | Isolation Forest + Autoencoder (fake-review detection) |
| `7_Recommendations.ipynb` | ALS collaborative filtering + cosine similarity + K-Means |